# Dataset Duplicate Analysis and Cleaning

Notebook-kan wuxuu falanqeynayaa `combined_dataset.csv`, wuxuu tirinayaa duplicate-yada, wuxuu soo bandhigayaa tusaalooyin, kadibna wuxuu kaydinayaa dataset nadiif ah.

In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

pd.set_option('display.max_colwidth', 150)

# Wuxuu shaqaynayaa haddii notebook-ka laga ordo project root-ka ama model folder-ka.
DATASET_PATH = Path('combined_dataset.csv')
if not DATASET_PATH.exists():
    DATASET_PATH = Path('model/combined_dataset.csv')

OUTPUT_PATH = DATASET_PATH.with_name('cleaned_dataset.csv')
DATASET_PATH, OUTPUT_PATH

In [ ]:
df = pd.read_csv(DATASET_PATH, encoding='utf-8', low_memory=False)

required_columns = {'text', 'category'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Columns-kan ayaa dataset-ka ka maqan: {sorted(missing_columns)}')

print(f'Tirada records-ka: {len(df):,}')
print(f'Tirada columns-ka: {df.shape[1]}')
print('\nMissing values:')
display(df.isna().sum().to_frame('missing_count'))
display(df.head())

## Duplicate analysis

Waxaan cabbiraynaa laba nooc: safaf gebi ahaan isku mid ah iyo qoraallo isku mid ah kaddib marka whitespace/case/Unicode la mideeyo.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value))
    value = re.sub(r'\s+', ' ', value).strip().casefold()
    return value

df['_normalized_text'] = df['text'].map(normalize_text)

exact_duplicate_count = int(df.duplicated(keep='first').sum())
text_duplicate_count = int(df.duplicated(subset=['_normalized_text'], keep='first').sum())
unique_text_count = int(df['_normalized_text'].nunique(dropna=False))

summary = pd.DataFrame({
    'metric': [
        'All records',
        'Exact duplicate rows',
        'Duplicate normalized texts',
        'Unique normalized texts',
    ],
    'count': [len(df), exact_duplicate_count, text_duplicate_count, unique_text_count],
})
summary['percentage_of_rows'] = (summary['count'] / len(df) * 100).round(2)
display(summary)

In [ ]:
# Tusaalooyinka qoraallada soo noqnoqda
duplicate_examples = (
    df[df.duplicated(subset=['_normalized_text'], keep=False)]
      .sort_values('_normalized_text')
      [['url', 'text', 'category']]
      .head(20)
)
display(duplicate_examples)

In [ ]:
# Hubi qoraal isku mid ah oo leh categories kala duwan.
label_counts = df.groupby('_normalized_text')['category'].nunique(dropna=False)
conflicting_keys = label_counts[label_counts > 1].index
conflicts = (
    df[df['_normalized_text'].isin(conflicting_keys)]
      .sort_values('_normalized_text')
      [['text', 'category']]
)
print(f'Qoraallada leh category is-khilaafsan: {len(conflicting_keys):,}')
display(conflicts.head(20))

## Ka saar duplicates-ka oo kaydi

Qoraal kasta waxaa laga reebayaa soo noqnoqoshada, waxaana la haynayaa record-kii ugu horreeyay. Dataset-ka asalka ah lama beddelayo.

In [ ]:
clean_df = (
    df.drop_duplicates(subset=['_normalized_text'], keep='first')
      .drop(columns=['_normalized_text'])
      .reset_index(drop=True)
)

removed_count = len(df) - len(clean_df)
clean_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f'Records-kii hore: {len(df):,}')
print(f'Duplicates laga saaray: {removed_count:,}')
print(f'Records-ka nadiifka ah: {len(clean_df):,}')
print(f'Waxaa lagu kaydiyey: {OUTPUT_PATH.resolve()}')

In [ ]:
# Final verification: faylka kaydsan dib u akhri oo xaqiiji.
saved_df = pd.read_csv(OUTPUT_PATH, encoding='utf-8', low_memory=False)
saved_normalized = saved_df['text'].map(normalize_text)
remaining_duplicates = int(saved_normalized.duplicated().sum())

assert len(saved_df) == len(clean_df), 'Tirada la kaydiyey sax ma aha.'
assert remaining_duplicates == 0, 'Duplicates ayaa wali ku jira output-ka.'

print(f'Verification passed: {len(saved_df):,} records, {remaining_duplicates} duplicates.')
display(saved_df['category'].value_counts(dropna=False).to_frame('count'))